# Cocktail RAG - LlamaParse Experiments

Πρώτα βήματα: parsing του "Drinks" by Jacques Straub (1914).

## Setup

In [10]:
# ═══════════════════════════════════════════════════════════════
# BOOTSTRAP — φορτώνει όλα από τα σωσμένα αρχεία
# ═══════════════════════════════════════════════════════════════

import os
import json
from pathlib import Path
from dotenv import load_dotenv
from anthropic import Anthropic
from llama_cloud import LlamaCloud

# 1. Environment variables
load_dotenv()
print("✅ Environment loaded")

# 2. API clients
anthropic_client = Anthropic()
llama_client = LlamaCloud()
print("✅ API clients ready")

# 3. Load enriched dataset
enriched_path = Path("data/enriched/straub_1914_enriched.json")
with open(enriched_path, "r", encoding="utf-8") as f:
    enriched_data = json.load(f)
enriched_list = list(enriched_data.values())
print(f"✅ Loaded {len(enriched_list)} enriched recipes")

# 4. Load clean chunks (πριν το enrichment)
chunks_path = Path("data/enriched/straub_1914_chunks.json")
with open(chunks_path, "r", encoding="utf-8") as f:
    clean = json.load(f)
print(f"✅ Loaded {len(clean)} clean chunks")

# 5. Verification
successful = [r for r in enriched_list if r.get('metadata') and isinstance(r['metadata'], dict)]
print(f"\n🎯 Ready to work!")
print(f"   • {len(successful)} recipes με metadata")
print(f"   • Anthropic API: ✅")
print(f"   • LlamaCloud API: ✅")

✅ Environment loaded
✅ API clients ready
✅ Loaded 549 enriched recipes
✅ Loaded 553 clean chunks

🎯 Ready to work!
   • 527 recipes με metadata
   • Anthropic API: ✅
   • LlamaCloud API: ✅


In [1]:
# Setup: load environment variables and initialize LlamaCloud client
import os
from pathlib import Path
from dotenv import load_dotenv
from llama_cloud import LlamaCloud

# Load .env file (θα διαβάσει το LLAMA_CLOUD_API_KEY)
load_dotenv()

# Verify το key φορτώθηκε
api_key = os.getenv("LLAMA_CLOUD_API_KEY")
if not api_key:
    raise ValueError("❌ LLAMA_CLOUD_API_KEY not found in .env file!")

print(f"✅ API Key loaded: {api_key[:8]}...{api_key[-4:]}")

# Initialize LlamaCloud client
client = LlamaCloud()
print("✅ LlamaCloud client initialized")

✅ API Key loaded: llx-9fZW...rbsn
✅ LlamaCloud client initialized


In [4]:
# Πρώτο parsing test με LlamaParse
from pathlib import Path

# Το PDF μας
pdf_path = Path("data/raw/Drinks_by_Jacques_Straub.pdf")

# Επιβεβαίωση ότι υπάρχει
if not pdf_path.exists():
    raise FileNotFoundError(f"❌ PDF not found at: {pdf_path.absolute()}")

print(f"✅ Found PDF: {pdf_path.name} ({pdf_path.stat().st_size / (1024*1024):.2f} MB)")

# Upload το PDF στο LlamaCloud
print("\n📤 Uploading PDF to LlamaCloud...")
with open(pdf_path, "rb") as f:
    uploaded_file = client.files.create(file=f, purpose="parse")

print(f"✅ Uploaded! File ID: {uploaded_file.id}")

✅ Found PDF: Drinks_by_Jacques_Straub.pdf (6.56 MB)

📤 Uploading PDF to LlamaCloud...


KeyboardInterrupt: 

In [ ]:
# Parse το PDF!
# Θα χρησιμοποιήσουμε tier "agentic" (καλή ισορροπία quality/cost)
# Είναι το recommended default

print("🔮 Parsing PDF... (αυτό μπορεί να πάρει 1-3 λεπτά)")
print("   Το LlamaParse δουλεύει σελίδα-σελίδα με vision AI...")

result = client.parsing.parse(
    file_id=uploaded_file.id,
    tier="agentic",
    version="latest",
    expand=["markdown"],
)

print(f"\n✅ Parsing complete!")
print(f"📄 Total pages: {len(result.markdown.pages)}")

In [5]:
# Δες την πρώτη σελίδα σε markdown
first_page = result.markdown.pages[0]

print("=" * 70)
print(f"📖 Page 1 - Markdown output")
print("=" * 70)
print(first_page.markdown)

NameError: name 'result' is not defined

In [ ]:
# Ας δούμε πολλές σελίδες μαζί για να βρούμε τις συνταγές
# Τυπώνουμε τις σελίδες 5, 15, 30, 50

for page_num in [5, 15, 30, 50]:
    page = result.markdown.pages[page_num - 1]  # 0-indexed
    print("=" * 70)
    print(f"📖 Page {page_num}")
    print("=" * 70)
    print(page.markdown)
    print("\n")

In [ ]:
# Save parsed markdown σε αρχείο - όλες οι σελίδες μαζί
from pathlib import Path

# Δημιούργησε φάκελο για parsed output
output_dir = Path("data/parsed")
output_dir.mkdir(parents=True, exist_ok=True)

# Συνένωσε όλες τις σελίδες με separator
all_pages_markdown = "\n\n---PAGE_BREAK---\n\n".join(
    [page.markdown for page in result.markdown.pages]
)

# Save
output_path = output_dir / "straub_1914.md"
output_path.write_text(all_pages_markdown, encoding="utf-8")

print(f"✅ Saved parsed markdown to: {output_path}")
print(f"📊 File size: {output_path.stat().st_size / 1024:.1f} KB")
print(f"📄 Total pages: {len(result.markdown.pages)}")
print(f"📝 Total characters: {len(all_pages_markdown):,}")

In [2]:
# Reload το parsed markdown από το αρχείο
from pathlib import Path

parsed_file = Path("data/parsed/straub_1914.md")

if not parsed_file.exists():
    raise FileNotFoundError(f"❌ Δεν βρέθηκε το αρχείο: {parsed_file.absolute()}")

# Διάβασε το markdown
full_markdown = parsed_file.read_text(encoding="utf-8")

# Σπάσε πίσω στις σελίδες
pages = full_markdown.split("\n\n---PAGE_BREAK---\n\n")

print(f"✅ Loaded parsed markdown from disk")
print(f"📄 Total pages: {len(pages)}")
print(f"📝 Total characters: {len(full_markdown):,}")
print(f"💾 File size: {parsed_file.stat().st_size / 1024:.1f} KB")

✅ Loaded parsed markdown from disk
📄 Total pages: 124
📝 Total characters: 171,776
💾 File size: 176.9 KB


In [6]:
# Πόσα ## headings υπάρχουν; (πόσες πιθανές συνταγές)
import re

# Ψάχνουμε γραμμές που ξεκινούν με '## '
h2_pattern = re.compile(r"^## (.+)$", re.MULTILINE)
h2_matches = h2_pattern.findall(full_markdown)

print(f"📊 Βρέθηκαν {len(h2_matches)} H2 headings (## ...)")
print(f"\n🔍 Πρώτα 20 headings:")
for i, heading in enumerate(h2_matches[:20], 1):
    print(f"  {i:2d}. {heading}")

print(f"\n🔍 Τελευταία 10 headings:")
for i, heading in enumerate(h2_matches[-10:], len(h2_matches) - 9):
    print(f"  {i:3d}. {heading}")

📊 Βρέθηκαν 559 H2 headings (## ...)

🔍 Πρώτα 20 headings:
   1. The Hotel Monthly Press
   2. How to Obtain Best Results
   3. A few lines from Oscar of The Waldorf
   4. Sauternes
   5. Rhine Wines
   6. Moselle Wines
   7. When to Serve Beverages
   8. Claret Cobbler
   9. Port Wine Cobbler
  10. Rhine Wine Cobbler
  11. Sherry Cobbler
  12. Whiskey Cobbler
  13. Absinthe Cocktail
  14. Adonis Cocktail
  15. Alaska Cocktail
  16. Alexander Cocktail
  17. Anderson Cocktail
  18. Antilles Cocktail
  19. Applejack Cocktail
  20. Ardsley Cocktail

🔍 Τελευταία 10 headings:
  550. Gin Toddy
  551. Kentucky Toddy
  552. Mint Toddy
  553. Peach Toddy
  554. Pendennis Toddy
  555. Rum Toddy
  556. Scotch Toddy
  557. Southern Toddy
  558. Whiskey Toddy
  559. for Hotel, Restaurant, Transportation Catering, Institution and Club Use


In [4]:
# Σπάσε το markdown σε chunks βάσει των ## headings
import re

def split_into_chunks(markdown_text: str) -> list[dict]:
    """
    Σπάει το markdown σε chunks με βάση τα ## headings.
    Κάθε chunk περιέχει το heading και το content μέχρι το επόμενο ##.
    """
    # Pattern: ## followed by heading, then everything until next ## or end
    pattern = re.compile(r"^## (.+?)$\n(.*?)(?=^## |\Z)", re.MULTILINE | re.DOTALL)

    chunks = []
    for match in pattern.finditer(markdown_text):
        heading = match.group(1).strip()
        content = match.group(2).strip()

        chunks.append({
            "name": heading,
            "content": content,
            "full_text": f"## {heading}\n\n{content}",
            "char_count": len(content),
        })

    return chunks

# Τρέξε το splitting
chunks = split_into_chunks(full_markdown)

print(f"✅ Δημιουργήθηκαν {len(chunks)} chunks")
print(f"\n📊 Στατιστικά:")
print(f"   Μέσο μέγεθος: {sum(c['char_count'] for c in chunks) / len(chunks):.0f} chars")
print(f"   Μικρότερο: {min(c['char_count'] for c in chunks)} chars")
print(f"   Μεγαλύτερο: {max(c['char_count'] for c in chunks)} chars")

# Δείξε το πρώτο chunk (intro)
print(f"\n{'='*70}")
print(f"📝 Chunk #1: {chunks[0]['name']}")
print(f"{'='*70}")
print(chunks[0]['content'][:300])  # Πρώτοι 300 χαρακτήρες
print("..." if chunks[0]['char_count'] > 300 else "")

✅ Δημιουργήθηκαν 559 chunks

📊 Στατιστικά:
   Μέσο μέγεθος: 283 chars
   Μικρότερο: 0 chars
   Μεγαλύτερο: 27213 chars

📝 Chunk #1: The Hotel Monthly Press
950 Merchandise Mart
Chicago, Ill.

---PAGE_BREAK---

# AUTHOR'S PREFACE



In [5]:
# Investigate προβληματικά chunks
print("🔍 Άδεια chunks (0 chars):")
empty_chunks = [i for i, c in enumerate(chunks) if c['char_count'] == 0]
print(f"   Πλήθος: {len(empty_chunks)}")
if empty_chunks:
    for idx in empty_chunks[:5]:
        print(f"   Chunk #{idx}: '{chunks[idx]['name']}'")

print(f"\n🔍 Μεγάλα chunks (>2000 chars):")
large_chunks = [(i, c) for i, c in enumerate(chunks) if c['char_count'] > 2000]
print(f"   Πλήθος: {len(large_chunks)}")
for idx, c in large_chunks[:5]:
    print(f"   Chunk #{idx}: '{c['name']}' - {c['char_count']} chars")

print(f"\n📏 Distribution:")
buckets = {"<50": 0, "50-200": 0, "200-500": 0, "500-1000": 0, "1000-2000": 0, ">2000": 0}
for c in chunks:
    n = c['char_count']
    if n < 50: buckets["<50"] += 1
    elif n < 200: buckets["50-200"] += 1
    elif n < 500: buckets["200-500"] += 1
    elif n < 1000: buckets["500-1000"] += 1
    elif n < 2000: buckets["1000-2000"] += 1
    else: buckets[">2000"] += 1

for bucket, count in buckets.items():
    bar = "█" * (count // 10)
    print(f"   {bucket:>12}: {count:4d} {bar}")

🔍 Άδεια chunks (0 chars):
   Πλήθος: 6
   Chunk #60: 'Colonial Cocktail, or'
   Chunk #97: 'Fourth Degree Cocktail'
   Chunk #177: 'Ojen Cocktail'
   Chunk #193: 'Perfect Cocktail'
   Chunk #344: 'FRAPPÉS'

🔍 Μεγάλα chunks (>2000 chars):
   Πλήθος: 7
   Chunk #2: 'A few lines from Oscar of The Waldorf' - 7989 chars
   Chunk #5: 'Moselle Wines' - 6707 chars
   Chunk #300: 'Grape Juice—(Without Liquor)' - 3073 chars
   Chunk #420: 'Jersey Flashlight' - 2958 chars
   Chunk #485: '1 GALLON' - 2686 chars

📏 Distribution:
            <50:   38 ███
         50-200:  457 █████████████████████████████████████████████
        200-500:   38 ███
       500-1000:    5 
      1000-2000:   14 █
          >2000:    7 


In [7]:
# Cleanup: αφαίρεση άδειων chunks + καθαρισμός page break markers
import re

def clean_chunks(chunks: list[dict]) -> list[dict]:
    """Καθαρίζει τα chunks: αφαιρεί άδεια & καθαρίζει page breaks."""
    cleaned = []
    for chunk in chunks:
        # Αφαίρεση page break markers
        cleaned_content = chunk["content"].replace("---PAGE_BREAK---", "").strip()
        # Αφαίρεση πολλαπλών κενών γραμμών
        cleaned_content = re.sub(r"\n{3,}", "\n\n", cleaned_content)

        # Skip αν είναι άδειο
        if not cleaned_content:
            continue

        cleaned.append({
            "name": chunk["name"],
            "content": cleaned_content,
            "full_text": f"## {chunk['name']}\n\n{cleaned_content}",
            "char_count": len(cleaned_content),
        })

    return cleaned

# Καθάρισε
clean = clean_chunks(chunks)

print(f"✅ Πριν: {len(chunks)} chunks")
print(f"✅ Μετά cleanup: {len(clean)} chunks")
print(f"❌ Αφαιρέθηκαν: {len(chunks) - len(clean)} άδεια")

# Δείξε ένα καθαρό cocktail recipe για επιβεβαίωση
# Πάμε στο chunk για "Absinthe Cocktail" (heading #13 στη λίστα)
absinthe = next((c for c in clean if c['name'] == 'Absinthe Cocktail'), None)
if absinthe:
    print(f"\n{'='*70}")
    print(f"📝 Παράδειγμα: {absinthe['name']}")
    print(f"{'='*70}")
    print(absinthe['content'])
    print(f"\n📊 Chars: {absinthe['char_count']}")

✅ Πριν: 559 chunks
✅ Μετά cleanup: 553 chunks
❌ Αφαιρέθηκαν: 6 άδεια

📝 Παράδειγμα: Absinthe Cocktail
3/4 jigger green absinthe.

1 dash orange and Angostura bitters.

1 dash anisette.

Shake well. Serve.

📊 Chars: 102


In [ ]:
# Save chunks σε JSON για μελλοντική χρήση
import json
from pathlib import Path

output_dir = Path("data/enriched")
output_dir.mkdir(parents=True, exist_ok=True)

chunks_path = output_dir / "straub_1914_chunks.json"

with open(chunks_path, "w", encoding="utf-8") as f:
    json.dump(clean, f, ensure_ascii=False, indent=2)

print(f"✅ Saved {len(clean)} chunks to: {chunks_path}")
print(f"💾 File size: {chunks_path.stat().st_size / 1024:.1f} KB")

In [7]:
# Reload environment variables (επειδή αλλάξαμε το .env)
from dotenv import load_dotenv
import os

load_dotenv(override=True)  # override για να ξαναδιαβάσει

# Verify και τα δύο keys
llama_key = os.getenv("LLAMA_CLOUD_API_KEY")
anthropic_key = os.getenv("ANTHROPIC_API_KEY")

print(f"LlamaCloud key: {llama_key[:8]}...{llama_key[-4:]}" if llama_key else "❌ Missing!")
print(f"Anthropic key:  {anthropic_key[:12]}...{anthropic_key[-4:]}" if anthropic_key else "❌ Missing!")

LlamaCloud key: llx-9fZW...rbsn
Anthropic key:  sk-ant-api03...nAAA


In [10]:
# Test call στον Claude API
from anthropic import Anthropic

anthropic_client = Anthropic()  # αυτόματα διαβάζει το ANTHROPIC_API_KEY

# Απλό test message
response = anthropic_client.messages.create(
    model="claude-haiku-4-5",  # γρήγορο & φθηνό μοντέλο
    max_tokens=100,
    messages=[
        {"role": "user", "content": "Say 'Hello, cocktails!' in Greek"}
    ]
)

print("✅ Claude responded:")
print(response.content[0].text)
print(f"\n📊 Tokens used: input={response.usage.input_tokens}, output={response.usage.output_tokens}")

✅ Claude responded:
Γεια σας, κοκτέιλ! (Yia sas, koktéil!)

Or more casually:

Γεία σου, κοκτέιλ! (Yia sou, koktéil!)

📊 Tokens used: input=17, output=66


In [8]:
# Φόρτωσε τα chunks από το JSON (αν δεν το έχεις κάνει ήδη)
import json
from pathlib import Path

chunks_path = Path("data/enriched/straub_1914_chunks.json")

with open(chunks_path, "r", encoding="utf-8") as f:
    clean = json.load(f)

print(f"✅ Loaded {len(clean)} chunks")

# Δείξε το Absinthe Cocktail
absinthe = next(c for c in clean if c['name'] == 'Absinthe Cocktail')
print(f"\n📝 Test recipe: {absinthe['name']}")
print("-" * 50)
print(absinthe['content'])

✅ Loaded 553 chunks

📝 Test recipe: Absinthe Cocktail
--------------------------------------------------
3/4 jigger green absinthe.

1 dash orange and Angostura bitters.

1 dash anisette.

Shake well. Serve.


In [9]:
# Το prompt μας για metadata extraction
ENRICHMENT_PROMPT = """You are analyzing a cocktail recipe from a 1914 bartender's guide.
Extract structured metadata as JSON.

Recipe name: {name}
Recipe content:
{content}

Extract the following fields and return ONLY valid JSON (no markdown, no explanation):

{{
  "type": "cocktail | punch | cobbler | cooler | fizz | toddy | frappé | cup | other | info",
  "base_spirit": "primary spirit (e.g., gin, whiskey, rum, brandy, vermouth, wine, none)",
  "all_ingredients": ["list of all ingredient names, normalized"],
  "flavor_profile": ["list of 2-4 flavor descriptors like bitter, sweet, herbal, citrus, fruity, spicy, dry, smoky"],
  "sweetness": "none | low | medium | high",
  "strength": "low | medium | high",
  "method": "shaken | stirred | built | blended | muddled | other",
  "glassware": "cocktail | rocks | highball | wine | punch | other",
  "ingredient_count": <integer>,
  "era_style": "classic pre-prohibition"
}}

If the recipe is actually just information/prose (not a drink), set type to "info" and other fields to null.
"""

# Format το prompt με τη συνταγή του Absinthe
test_prompt = ENRICHMENT_PROMPT.format(
    name=absinthe['name'],
    content=absinthe['content']
)

# Καλέσε τον Claude
print("🧠 Calling Claude for enrichment...")

response = anthropic_client.messages.create(
    model="claude-haiku-4-5",
    max_tokens=500,
    messages=[
        {"role": "user", "content": test_prompt}
    ]
)

# Δες το raw response
raw_output = response.content[0].text
print("\n📥 Claude's response:")
print(raw_output)

print(f"\n📊 Tokens: input={response.usage.input_tokens}, output={response.usage.output_tokens}")

🧠 Calling Claude for enrichment...


NameError: name 'anthropic_client' is not defined

In [13]:
import json
import re

def parse_claude_json(raw_text: str) -> dict:
    """
    Καθαρίζει το output του Claude από markdown code blocks και το parse-άρει σε dict.
    """
    # Αφαίρεσε markdown code fences αν υπάρχουν
    cleaned = raw_text.strip()
    
    # Remove ```json ... ``` wrapper
    cleaned = re.sub(r"^```(?:json)?\s*\n", "", cleaned)
    cleaned = re.sub(r"\n```\s*$", "", cleaned)
    
    return json.loads(cleaned)


# Parse το output του Absinthe
metadata = parse_claude_json(raw_output)

print("✅ Successfully parsed to dict!")
print(f"\n🔍 Metadata για: {absinthe['name']}")
print("-" * 50)
for key, value in metadata.items():
    print(f"  {key:20s}: {value}")

print(f"\n📊 Type of result: {type(metadata).__name__}")
print(f"📊 Base spirit is: '{metadata['base_spirit']}'")

✅ Successfully parsed to dict!

🔍 Metadata για: Absinthe Cocktail
--------------------------------------------------
  type                : cocktail
  base_spirit         : absinthe
  all_ingredients     : ['green absinthe', 'orange bitters', 'angostura bitters', 'anisette']
  flavor_profile      : ['herbal', 'bitter', 'anise', 'citrus']
  sweetness           : low
  strength            : high
  method              : shaken
  glassware           : cocktail
  ingredient_count    : 4
  era_style           : classic pre-prohibition

📊 Type of result: dict
📊 Base spirit is: 'absinthe'


In [15]:
# Test enrichment σε 5 διαφορετικά chunks
import time

# Πάρε 5 διαφορετικά είδη chunks
test_indices = [
    2,    # "A few lines from Oscar of The Waldorf" - PROSE
    7,    # "Claret Cobbler" - cobbler
    12,   # "Absinthe Cocktail" - cocktail
    100,  # τυχαία συνταγή στη μέση
    350,  # τυχαία συνταγή αργότερα
]

results = []

for i, idx in enumerate(test_indices, 1):
    chunk = clean[idx]
    print("\n" + "=" * 70)
    print(f"[{i}/{len(test_indices)}] {chunk['name']}")
    print("=" * 70)
    preview = chunk['content'][:200]
    suffix = "..." if len(chunk['content']) > 200 else ""
    print(f"📝 Content: {preview}{suffix}")

    prompt = ENRICHMENT_PROMPT.format(
        name=chunk['name'],
        content=chunk['content']
    )

    response = anthropic_client.messages.create(
        model="claude-haiku-4-5",
        max_tokens=500,
        messages=[{"role": "user", "content": prompt}]
    )

    try:
        metadata = parse_claude_json(response.content[0].text)
        results.append({
            "name": chunk['name'],
            "metadata": metadata,
            "tokens_in": response.usage.input_tokens,
            "tokens_out": response.usage.output_tokens,
        })
        print(f"\n✅ Type: {metadata.get('type')}")
        print(f"   Base spirit: {metadata.get('base_spirit')}")
        print(f"   Flavors: {metadata.get('flavor_profile')}")
        print(f"   Tokens: in={response.usage.input_tokens}, out={response.usage.output_tokens}")
    except json.JSONDecodeError as e:
        print(f"❌ Failed to parse JSON: {e}")
        print(f"Raw output: {response.content[0].text[:300]}")

    time.sleep(0.5)

# Στατιστικά
total_input = sum(r['tokens_in'] for r in results)
total_output = sum(r['tokens_out'] for r in results)
cost = (total_input * 0.80 + total_output * 4.00) / 1_000_000
extrapolated = cost * 553 / len(test_indices)

print("\n" + "=" * 70)
print("📊 SUMMARY")
print("=" * 70)
print(f"✅ Successfully enriched: {len(results)}/{len(test_indices)}")
print(f"💰 Total tokens: {total_input} in + {total_output} out")
print(f"💰 Estimated cost for 5 recipes: ${cost:.5f}")
print(f"💰 Extrapolated for 553 recipes: ${extrapolated:.2f}")


[1/5] A few lines from Oscar of The Waldorf
📝 Content: My friend, Jacques Straub, who wrote the book Drinks, gave to the world a classic in wholesome temperance beverages. Where its precepts are followed there is true temperance.

It is a book appreciated...

✅ Type: info
   Base spirit: None
   Flavors: None
   Tokens: in=2148, out=94

[2/5] Claret Cobbler
📝 Content: Fill goblet with fine ice.
1/2 jigger syrup.
1 1/2 jigger claret.
Stir; decorate with fruit.

✅ Type: cobbler
   Base spirit: wine
   Flavors: ['sweet', 'fruity', 'refreshing']
   Tokens: in=353, out=127

[3/5] Absinthe Cocktail
📝 Content: 3/4 jigger green absinthe.

1 dash orange and Angostura bitters.

1 dash anisette.

Shake well. Serve.

✅ Type: cocktail
   Base spirit: absinthe
   Flavors: ['herbal', 'bitter', 'anise', 'citrus']
   Tokens: in=354, out=144

[4/5] Gibson Cocktail
📝 Content: 1/2 jigger French vermouth.

1/2 jigger dry gin.

Stir, strain and serve.

✅ Type: cocktail
   Base spirit: gin
   Flavors: ['dry

In [16]:
# BULK ENRICHMENT - όλες τις 553 συνταγές
import time
from pathlib import Path
import json

# Path για αυτόματο save
enriched_path = Path("data/enriched/straub_1914_enriched.json")

# Φόρτωσε ήδη enriched (αν υπάρχει από προηγούμενη διακοπή)
already_enriched = {}
if enriched_path.exists():
    with open(enriched_path, "r", encoding="utf-8") as f:
        already_enriched = json.load(f)
    print(f"📂 Found {len(already_enriched)} already enriched — will skip these")
else:
    print("📂 Starting fresh — no previous enrichment found")

# Στατιστικά
total_input_tokens = 0
total_output_tokens = 0
successful = 0
failed = 0
skipped = 0

# Function για save
def save_progress(data, path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)

# Χρησιμοποιούμε το name ως unique key
enriched_data = dict(already_enriched)  # copy

print(f"\n🚀 Starting enrichment of {len(clean)} recipes...")
print(f"⏱️  Estimated time: ~10-15 minutes")
print(f"💰 Estimated cost: ~$0.60")
print("=" * 70)

start_time = time.time()

for i, chunk in enumerate(clean, 1):
    name = chunk['name']
    
    # Skip αν είναι ήδη enriched
    if name in enriched_data:
        skipped += 1
        continue
    
    # Progress every 10 recipes
    if i % 10 == 0 or i == 1:
        elapsed = time.time() - start_time
        rate = (i - skipped) / elapsed if elapsed > 0 else 0
        cost_so_far = (total_input_tokens * 0.80 + total_output_tokens * 4.00) / 1_000_000
        print(f"[{i:3d}/{len(clean)}] ✅ {successful} success | ❌ {failed} failed | "
              f"⏭️ {skipped} skipped | 💰 ${cost_so_far:.3f} | ⚡ {rate:.1f} rec/s")
    
    # Format prompt
    prompt = ENRICHMENT_PROMPT.format(
        name=chunk['name'],
        content=chunk['content']
    )
    
    try:
        # Call Claude
        response = anthropic_client.messages.create(
            model="claude-haiku-4-5",
            max_tokens=500,
            messages=[{"role": "user", "content": prompt}]
        )
        
        # Parse response
        metadata = parse_claude_json(response.content[0].text)
        
        # Store enriched chunk
        enriched_data[name] = {
            "name": name,
            "content": chunk['content'],
            "full_text": chunk['full_text'],
            "char_count": chunk['char_count'],
            "metadata": metadata,
        }
        
        successful += 1
        total_input_tokens += response.usage.input_tokens
        total_output_tokens += response.usage.output_tokens
        
    except json.JSONDecodeError as e:
        # JSON parse failed — save raw response for later inspection
        enriched_data[name] = {
            "name": name,
            "content": chunk['content'],
            "full_text": chunk['full_text'],
            "char_count": chunk['char_count'],
            "metadata": None,
            "error": f"JSON parse: {str(e)[:100]}",
            "raw_response": response.content[0].text[:500] if 'response' in dir() else None,
        }
        failed += 1
        print(f"   ⚠️  JSON error for '{name}'")
        
    except Exception as e:
        enriched_data[name] = {
            "name": name,
            "content": chunk['content'],
            "full_text": chunk['full_text'],
            "char_count": chunk['char_count'],
            "metadata": None,
            "error": f"API error: {str(e)[:100]}",
        }
        failed += 1
        print(f"   ⚠️  API error for '{name}': {str(e)[:100]}")
    
    # Auto-save κάθε 50 recipes
    if i % 50 == 0:
        save_progress(enriched_data, enriched_path)
        print(f"   💾 Auto-saved progress at recipe {i}")
    
    # Rate limiting - μικρό sleep
    time.sleep(0.3)

# Final save
save_progress(enriched_data, enriched_path)

# Final stats
elapsed = time.time() - start_time
final_cost = (total_input_tokens * 0.80 + total_output_tokens * 4.00) / 1_000_000

print("\n" + "=" * 70)
print("🎉 BULK ENRICHMENT COMPLETE!")
print("=" * 70)
print(f"⏱️  Total time: {elapsed / 60:.1f} minutes")
print(f"✅ Successful: {successful}")
print(f"❌ Failed: {failed}")
print(f"⏭️  Skipped (already done): {skipped}")
print(f"📊 Total tokens: {total_input_tokens:,} in + {total_output_tokens:,} out")
print(f"💰 Total cost: ${final_cost:.4f}")
print(f"💾 Saved to: {enriched_path}")

📂 Starting fresh — no previous enrichment found

🚀 Starting enrichment of 553 recipes...
⏱️  Estimated time: ~10-15 minutes
💰 Estimated cost: ~$0.60
[  1/553] ✅ 0 success | ❌ 0 failed | ⏭️ 0 skipped | 💰 $0.000 | ⚡ 2381.8 rec/s
[ 10/553] ✅ 9 success | ❌ 0 failed | ⏭️ 0 skipped | 💰 $0.010 | ⚡ 0.6 rec/s
[ 20/553] ✅ 19 success | ❌ 0 failed | ⏭️ 0 skipped | 💰 $0.018 | ⚡ 0.5 rec/s
   ⚠️  JSON error for 'Astoria Cocktail'
[ 30/553] ✅ 28 success | ❌ 1 failed | ⏭️ 0 skipped | 💰 $0.025 | ⚡ 0.5 rec/s
   ⚠️  JSON error for 'Bobbie Burns Cocktail (For Two)'
[ 40/553] ✅ 37 success | ❌ 2 failed | ⏭️ 0 skipped | 💰 $0.033 | ⚡ 0.5 rec/s
[ 50/553] ✅ 47 success | ❌ 2 failed | ⏭️ 0 skipped | 💰 $0.041 | ⚡ 0.5 rec/s
   💾 Auto-saved progress at recipe 50
[ 60/553] ✅ 57 success | ❌ 2 failed | ⏭️ 0 skipped | 💰 $0.050 | ⚡ 0.5 rec/s
[ 70/553] ✅ 67 success | ❌ 2 failed | ⏭️ 0 skipped | 💰 $0.058 | ⚡ 0.5 rec/s
[ 80/553] ✅ 77 success | ❌ 2 failed | ⏭️ 0 skipped | 💰 $0.066 | ⚡ 0.5 rec/s
[ 90/553] ✅ 87 success | ❌ 2 fa

In [18]:
# Explore το enriched dataset (safe version)
import json
from collections import Counter
from pathlib import Path

# Φόρτωσε το enriched
enriched_path = Path("data/enriched/straub_1914_enriched.json")
with open(enriched_path, "r", encoding="utf-8") as f:
    enriched_data = json.load(f)

# Convert σε list
enriched_list = list(enriched_data.values())

# Filter: μόνο όσα έχουν metadata ΚΑΙ είναι dict
successful = [
    r for r in enriched_list 
    if r.get('metadata') is not None and isinstance(r['metadata'], dict)
]
failed = [r for r in enriched_list if not r.get('metadata')]
weird = [
    r for r in enriched_list 
    if r.get('metadata') is not None and not isinstance(r['metadata'], dict)
]

print(f"📊 Dataset Overview")
print(f"=" * 60)
print(f"Total chunks: {len(enriched_list)}")
print(f"✅ With dict metadata: {len(successful)}")
print(f"⚠️  Weird metadata (list instead of dict): {len(weird)}")
print(f"❌ Failed (no metadata): {len(failed)}")

# Show a weird example
if weird:
    print(f"\n🔍 Example of weird metadata:")
    print(f"   Name: {weird[0]['name']}")
    print(f"   Type: {type(weird[0]['metadata']).__name__}")
    weird_preview = str(weird[0]['metadata'])[:200]
    print(f"   Preview: {weird_preview}...")

# Distribution by type
print(f"\n📊 Recipes by Type")
print(f"-" * 60)
types = Counter(r['metadata'].get('type') or 'unknown' for r in successful)
for type_name, count in types.most_common():
    bar = "█" * (count // 5)
    print(f"  {str(type_name):15s}: {count:4d} {bar}")

# Distribution by base spirit
print(f"\n🍾 Recipes by Base Spirit (top 15)")
print(f"-" * 60)
spirits = Counter(r['metadata'].get('base_spirit') or 'unknown' for r in successful)
for spirit, count in spirits.most_common(15):
    bar = "█" * (count // 3)
    print(f"  {str(spirit):20s}: {count:4d} {bar}")

# Distribution by method
print(f"\n🥄 Preparation Methods")
print(f"-" * 60)
methods = Counter(r['metadata'].get('method') or 'unknown' for r in successful)
for method, count in methods.most_common():
    bar = "█" * (count // 5)
    print(f"  {str(method):15s}: {count:4d} {bar}")

# Distribution by strength
print(f"\n💪 Strength Distribution")
print(f"-" * 60)
strengths = Counter(r['metadata'].get('strength') or 'unknown' for r in successful)
for strength, count in strengths.most_common():
    bar = "█" * (count // 5)
    print(f"  {str(strength):15s}: {count:4d} {bar}")

📊 Dataset Overview
Total chunks: 549
✅ With dict metadata: 527
⚠️  Weird metadata (list instead of dict): 4
❌ Failed (no metadata): 18

🔍 Example of weird metadata:
   Name: Ojen Cocktail—(New Orleans Style)
   Type: list
   Preview: [{'type': 'cocktail', 'base_spirit': 'none', 'all_ingredients': ['Ojen', 'Peychaud bitters'], 'flavor_profile': ['bitter', 'herbal', 'spicy'], 'sweetness': 'low', 'strength': 'high', 'method': 'shaken...

📊 Recipes by Type
------------------------------------------------------------
  cocktail       :  281 ████████████████████████████████████████████████████████
  cooler         :   49 █████████
  fizz           :   48 █████████
  punch          :   36 ███████
  toddy          :   27 █████
  frappé         :   22 ████
  info           :   20 ████
  other          :   19 ███
  highball       :    8 █
  cup            :    6 █
  cobbler        :    5 █
  daisy          :    3 
  sour           :    2 
  fix            :    1 

🍾 Recipes by Base Spirit (top 1

In [11]:
# Reload environment variables μετά την αλλαγή του .env
from dotenv import load_dotenv
import os

load_dotenv(override=True)

# Verify και τα τρία keys
llama_key = os.getenv("LLAMA_CLOUD_API_KEY")
anthropic_key = os.getenv("ANTHROPIC_API_KEY")
pinecone_key = os.getenv("PINECONE_API_KEY")

print(f"LlamaCloud: {llama_key[:8]}...{llama_key[-4:]}" if llama_key else "❌ LlamaCloud missing")
print(f"Anthropic:  {anthropic_key[:12]}...{anthropic_key[-4:]}" if anthropic_key else "❌ Anthropic missing")
print(f"Pinecone:   {pinecone_key[:12]}...{pinecone_key[-4:]}" if pinecone_key else "❌ Pinecone missing")

LlamaCloud: llx-9fZW...rbsn
Anthropic:  sk-ant-api03...nAAA
Pinecone:   pcsk_SVJuc_S...w2S3


In [12]:
# Connect στο Pinecone και δες τι υπάρχει
from pinecone import Pinecone

pc = Pinecone()  # αυτόματα διαβάζει το PINECONE_API_KEY

# List existing indexes
existing_indexes = pc.list_indexes()

print("✅ Connected to Pinecone!")
print(f"\n📋 Existing indexes: {len(existing_indexes)}")
for idx in existing_indexes:
    print(f"   • {idx.name} (dimension={idx.dimension})")

if not existing_indexes:
    print("   (None yet — we'll create one!)")

✅ Connected to Pinecone!

📋 Existing indexes: 0
   (None yet — we'll create one!)


In [13]:
# Create Pinecone index για τα cocktails
from pinecone import ServerlessSpec

INDEX_NAME = "cocktail-rag"
DIMENSION = 1536  # OpenAI text-embedding-3-small
METRIC = "cosine"

# Check αν υπάρχει ήδη (αποφυγή error)
existing_names = [idx.name for idx in pc.list_indexes()]

if INDEX_NAME in existing_names:
    print(f"⚠️  Index '{INDEX_NAME}' already exists — using it")
else:
    print(f"🌲 Creating index '{INDEX_NAME}'...")
    pc.create_index(
        name=INDEX_NAME,
        dimension=DIMENSION,
        metric=METRIC,
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1"
        )
    )
    print(f"✅ Index created!")

# Connect στο index
index = pc.Index(INDEX_NAME)

# Δες τα stats
stats = index.describe_index_stats()
print(f"\n📊 Index stats:")
print(f"   Dimension: {DIMENSION}")
print(f"   Metric: {METRIC}")
print(f"   Total vectors: {stats.get('total_vector_count', 0)}")

🌲 Creating index 'cocktail-rag'...
✅ Index created!

📊 Index stats:
   Dimension: 1536
   Metric: cosine
   Total vectors: 0


In [14]:
# Reload και check και τα 4 keys
from dotenv import load_dotenv
import os

load_dotenv(override=True)

keys = {
    "LlamaCloud": os.getenv("LLAMA_CLOUD_API_KEY"),
    "Anthropic":  os.getenv("ANTHROPIC_API_KEY"),
    "Pinecone":   os.getenv("PINECONE_API_KEY"),
    "OpenAI":     os.getenv("OPENAI_API_KEY"),
}

for name, key in keys.items():
    if key:
        print(f"✅ {name:12s}: {key[:12]}...{key[-4:]}")
    else:
        print(f"❌ {name:12s}: MISSING")

✅ LlamaCloud  : llx-9fZWq1vo...rbsn
✅ Anthropic   : sk-ant-api03...nAAA
✅ Pinecone    : pcsk_SVJuc_S...w2S3
✅ OpenAI      : sk-proj-PX9Q...1ssA


In [15]:
# Πρώτο embedding test - δες τι είναι ένα embedding
from openai import OpenAI

openai_client = OpenAI()  # αυτόματα διαβάζει το OPENAI_API_KEY

# Πάρε τον Absinthe Cocktail
absinthe = next(r for r in enriched_list if r['name'] == 'Absinthe Cocktail')

# Το κείμενο που θα κάνουμε embed
text_to_embed = absinthe['full_text']

print("📝 Κείμενο που θα γίνει embed:")
print("-" * 60)
print(text_to_embed[:300])
print("-" * 60)

# Create embedding!
print("\n🧠 Calling OpenAI embeddings API...")
response = openai_client.embeddings.create(
    model="text-embedding-3-small",
    input=text_to_embed
)

# Extract το vector
embedding = response.data[0].embedding

print(f"\n✅ Embedding created!")
print(f"📊 Vector dimension: {len(embedding)}")
print(f"📊 Type: {type(embedding).__name__}")
print(f"\n🔍 Πρώτα 10 numbers:")
print(embedding[:10])
print(f"\n🔍 Τελευταία 5 numbers:")
print(embedding[-5:])
print(f"\n💰 Tokens used: {response.usage.total_tokens}")

📝 Κείμενο που θα γίνει embed:
------------------------------------------------------------
## Absinthe Cocktail

3/4 jigger green absinthe.

1 dash orange and Angostura bitters.

1 dash anisette.

Shake well. Serve.
------------------------------------------------------------

🧠 Calling OpenAI embeddings API...

✅ Embedding created!
📊 Vector dimension: 1536
📊 Type: list

🔍 Πρώτα 10 numbers:
[-0.0833740234375, -0.01727294921875, -0.0097198486328125, -0.023529052734375, -0.00905609130859375, -0.0217437744140625, -0.0207061767578125, 0.01520538330078125, 0.0254974365234375, -0.0027561187744140625]

🔍 Τελευταία 5 numbers:
[0.0137786865234375, 0.0272064208984375, -0.041290283203125, -0.004566192626953125, 0.01525115966796875]

💰 Tokens used: 38


In [17]:
# Βρες ποια chunks είναι πάρα πολύ μεγάλα
import tiktoken

# tiktoken μετράει tokens όπως το OpenAI
encoding = tiktoken.encoding_for_model("text-embedding-3-small")

print("🔍 Ελέγχω ποια chunks είναι πολύ μεγάλα...")
print("-" * 60)

problematic = []
for i, r in enumerate(recipes_to_index):
    tokens = len(encoding.encode(r['full_text']))
    if tokens > 8000:  # Άφησε buffer
        problematic.append((i, r['name'], tokens))

print(f"\n⚠️  Chunks που ξεπερνούν 8000 tokens: {len(problematic)}")
for i, name, tokens in problematic:
    print(f"   Index {i}: '{name}' - {tokens:,} tokens")

# Επίσης δείξε top 10 μεγαλύτερα για επισκόπηση
print(f"\n📊 Top 10 μεγαλύτερα chunks:")
all_sizes = [(i, r['name'], len(encoding.encode(r['full_text']))) for i, r in enumerate(recipes_to_index)]
all_sizes.sort(key=lambda x: -x[2])
for i, name, tokens in all_sizes[:10]:
    marker = "🔴" if tokens > 8000 else "🟡" if tokens > 3000 else "🟢"
    print(f"   {marker} Index {i}: '{name}' - {tokens:,} tokens")

🔍 Ελέγχω ποια chunks είναι πολύ μεγάλα...
------------------------------------------------------------

⚠️  Chunks που ξεπερνούν 8000 tokens: 1
   Index 525: 'Whiskey Toddy' - 10,985 tokens

📊 Top 10 μεγαλύτερα chunks:
   🔴 Index 525: 'Whiskey Toddy' - 10,985 tokens
   🟡 Index 526: 'for Hotel, Restaurant, Transportation Catering, Institution and Club Use' - 5,725 tokens
   🟢 Index 2: 'A few lines from Oscar of The Waldorf' - 1,713 tokens
   🟢 Index 5: 'Moselle Wines' - 1,476 tokens
   🟢 Index 6: 'When to Serve Beverages' - 431 tokens
   🟢 Index 3: 'Sauternes' - 298 tokens
   🟢 Index 4: 'Rhine Wines' - 200 tokens
   🟢 Index 1: 'How to Obtain Best Results' - 163 tokens
   🟢 Index 262: 'Tom Collins' - 148 tokens
   🟢 Index 287: 'Cider Cup—(Without Liquor)' - 113 tokens


In [18]:
# Δες τι περιέχει το προβληματικό chunk
weird_chunk = recipes_to_index[525]

print(f"📖 Chunk: {weird_chunk['name']}")
print(f"📊 Chars: {weird_chunk['char_count']:,}")
print(f"\n{'='*70}")
print("ΠΡΩΤΟΙ 500 chars:")
print("="*70)
print(weird_chunk['full_text'][:500])

print(f"\n{'='*70}")
print("ΤΕΛΕΥΤΑΙΟΙ 500 chars:")
print("="*70)
print(weird_chunk['full_text'][-500:])

📖 Chunk: Whiskey Toddy
📊 Chars: 26,907

ΠΡΩΤΟΙ 500 chars:
## Whiskey Toddy

Crush ½ lump of sugar with a little water in old fashion glass.

1 jigger bourbon.

1 lemon peel. Stir.

# INDEX

**ABSINTHE**
Absinthe cocktail 17
drip 67
frappe 61
Adalor cup 48
Adonis cocktail 17
Alaska cocktail 17
Albern cocktail 25
Ale Sangaree 67
Alexander cocktail 17
American beauty punch 81
grog 63
grog, hot 63
Amer. Picon highball 62
pouffle 67
pouffle fizz 55
sour 92
Ammonia and Seltzer 67
Anderson cocktail 17
Angel blush 67
dream 67
kiss 67
tip 67
Angostura fizz 55
g

ΤΕΛΕΥΤΑΙΟΙ 500 chars:
 Starch

<u>Note</u>: Corn Starch should be added to the batter in original recipe (see previous page) to prevent curdling of egg albumin.
Quantity of starch for larger recipe —

# MEMORANDA

# MEMORANDA

Brownies Recipe
5 Eggs
Separate ~~rounded~~
1 tsp cornstarch for each egg
rounded
add sugar [illegible] to make
thick batter
Beat up whites - sugar to
thicken
Fold white into yolk

1 Tsp butter in the mug
Fill wit

In [19]:
# Truncate τα chunks που ξεπερνούν το token limit
import tiktoken

encoding = tiktoken.encoding_for_model("text-embedding-3-small")
MAX_TOKENS = 8000

def truncate_to_tokens(text: str, max_tokens: int) -> tuple[str, int]:
    tokens = encoding.encode(text)
    if len(tokens) <= max_tokens:
        return text, len(tokens)
    truncated_tokens = tokens[:max_tokens]
    truncated_text = encoding.decode(truncated_tokens)
    return truncated_text, len(truncated_tokens)


truncated_count = 0
for r in recipes_to_index:
    original_text = r['full_text']
    truncated_text, token_count = truncate_to_tokens(original_text, MAX_TOKENS)
    
    if len(truncated_text) < len(original_text):
        r['full_text'] = truncated_text
        r['was_truncated'] = True
        truncated_count += 1
        print(f"✂️  Truncated '{r['name']}': {len(original_text):,} → {len(truncated_text):,} chars ({token_count:,} tokens)")

print(f"\n✅ Truncated {truncated_count} chunks")
print(f"📦 Ready to embed {len(recipes_to_index)} recipes")

✂️  Truncated 'Whiskey Toddy': 26,925 → 19,473 chars (8,000 tokens)

✅ Truncated 1 chunks
📦 Ready to embed 527 recipes


In [20]:
# BATCH EMBEDDINGS + UPLOAD στο Pinecone
import time
from tqdm.notebook import tqdm

# Παίρνουμε μόνο τα successful (με metadata dict)
recipes_to_index = [
    r for r in enriched_list 
    if r.get('metadata') and isinstance(r['metadata'], dict)
]

print(f"📦 Θα ανεβάσουμε {len(recipes_to_index)} συνταγές στο Pinecone")

# Batch size - OpenAI δέχεται μεγάλα batches (μέχρι 2048)
BATCH_SIZE = 100

# Stats
total_tokens = 0
total_uploaded = 0

# Progress bar
for batch_start in tqdm(range(0, len(recipes_to_index), BATCH_SIZE), desc="Batches"):
    batch_end = min(batch_start + BATCH_SIZE, len(recipes_to_index))
    batch = recipes_to_index[batch_start:batch_end]
    
    # 1. Prepare texts για embedding
    texts = [r['full_text'] for r in batch]
    
    # 2. Call OpenAI για batch embeddings
    response = openai_client.embeddings.create(
        model="text-embedding-3-small",
        input=texts
    )
    
    total_tokens += response.usage.total_tokens
    
    # 3. Prepare vectors για Pinecone upload
    vectors = []
    for i, recipe in enumerate(batch):
        embedding = response.data[i].embedding
        
        # Metadata για κάθε vector (θα το χρησιμοποιήσουμε για filtering)
        meta = recipe['metadata']
        pinecone_metadata = {
            "name": recipe['name'],
            "type": str(meta.get('type') or 'unknown'),
            "base_spirit": str(meta.get('base_spirit') or 'unknown'),
            "method": str(meta.get('method') or 'unknown'),
            "strength": str(meta.get('strength') or 'unknown'),
            "sweetness": str(meta.get('sweetness') or 'unknown'),
            "ingredient_count": int(meta.get('ingredient_count') or 0),
            "flavor_profile": meta.get('flavor_profile') or [],
            "text": recipe['full_text'][:1000],  # Store για retrieval
        }
        
        # ID - χρησιμοποιούμε το όνομα σε safe format
        vector_id = f"recipe_{batch_start + i}"
        
        vectors.append({
            "id": vector_id,
            "values": embedding,
            "metadata": pinecone_metadata
        })
    
    # 4. Upsert στο Pinecone
    index.upsert(vectors=vectors)
    total_uploaded += len(vectors)
    
    # Μικρή παύση για rate limits
    time.sleep(0.1)

# Final stats
cost = (total_tokens / 1_000_000) * 0.02
print(f"\n🎉 Complete!")
print(f"✅ Uploaded: {total_uploaded} recipes")
print(f"💰 Total tokens: {total_tokens:,}")
print(f"💰 Total cost: ${cost:.5f}")

# Verify Pinecone stats
time.sleep(3)  # Wait για indexing
stats = index.describe_index_stats()
print(f"\n📊 Pinecone stats:")
print(f"   Total vectors: {stats.get('total_vector_count', 0)}")

📦 Θα ανεβάσουμε 527 συνταγές στο Pinecone


Batches:   0%|          | 0/6 [00:00<?, ?it/s]


🎉 Complete!
✅ Uploaded: 527 recipes
💰 Total tokens: 38,051
💰 Total cost: $0.00076

📊 Pinecone stats:
   Total vectors: 527


In [21]:
# ΤΟ ΠΡΩΤΟ ΣΟΥ SEMANTIC QUERY!
def search_cocktails(query: str, top_k: int = 5):
    """
    Ψάχνει στη βάση cocktails με semantic search.
    """
    # 1. Κάνε embed το query
    query_embedding = openai_client.embeddings.create(
        model="text-embedding-3-small",
        input=query
    ).data[0].embedding
    
    # 2. Query το Pinecone
    results = index.query(
        vector=query_embedding,
        top_k=top_k,
        include_metadata=True
    )
    
    # 3. Πρέζενταρε τα αποτελέσματα
    print(f"\n🔍 Query: '{query}'")
    print(f"{'='*70}")
    for i, match in enumerate(results['matches'], 1):
        meta = match['metadata']
        print(f"\n{i}. {meta['name']} (score: {match['score']:.3f})")
        print(f"   Type: {meta['type']} | Base: {meta['base_spirit']} | Strength: {meta['strength']}")
        print(f"   Method: {meta['method']} | Sweetness: {meta['sweetness']}")
        if meta.get('flavor_profile'):
            print(f"   Flavors: {', '.join(meta['flavor_profile'])}")
    return results


# Test με ένα απλό query
results = search_cocktails("something with gin and herbal notes")


🔍 Query: 'something with gin and herbal notes'

1. Gin Daisy (score: 0.535)
   Type: daisy | Base: gin | Strength: medium
   Method: built | Sweetness: medium
   Flavors: citrus, fruity, sweet

2. Gin Toddy (score: 0.525)
   Type: toddy | Base: gin | Strength: unknown
   Method: unknown | Sweetness: unknown

3. Ginger Daisy (score: 0.515)
   Type: cocktail | Base: gin | Strength: high
   Method: shaken | Sweetness: medium
   Flavors: citrus, sweet, herbal, fruity

4. Polly (score: 0.497)
   Type: fizz | Base: gin | Strength: medium
   Method: shaken | Sweetness: high
   Flavors: fruity, sweet, citrus, effervescent

5. Gin Sour (score: 0.496)
   Type: cocktail | Base: gin | Strength: high
   Method: shaken | Sweetness: low
   Flavors: citrus, sweet, dry


In [22]:
# Δοκίμασε διαφορετικά queries
queries = [
    "refreshing summer drink with citrus",
    "strong whiskey cocktail before dinner",
    "champagne cocktail for celebration",
    "hot drink for cold winter night",
    "sweet dessert cocktail with cream",
]

for q in queries:
    search_cocktails(q, top_k=3)
    print()  # blank line


🔍 Query: 'refreshing summer drink with citrus'

1. Fruit Lemonade (score: 0.582)
   Type: punch | Base: none | Strength: low
   Method: built | Sweetness: high
   Flavors: fruity, citrus, sweet, refreshing

2. Sea Side Cooler (score: 0.579)
   Type: cooler | Base: none | Strength: low
   Method: built | Sweetness: high
   Flavors: sweet, citrus, refreshing

3. Orangeade (score: 0.576)
   Type: cooler | Base: none | Strength: low
   Method: shaken | Sweetness: medium
   Flavors: citrus, sweet, refreshing


🔍 Query: 'strong whiskey cocktail before dinner'

1. Whiskey Cobbler (score: 0.590)
   Type: cobbler | Base: whiskey | Strength: medium
   Method: built | Sweetness: medium
   Flavors: sweet, citrus, fruity, smooth

2. Whiskey Cocktail (score: 0.577)
   Type: cocktail | Base: whiskey | Strength: high
   Method: stirred | Sweetness: low
   Flavors: bitter, sweet, citrus, spicy

3. Black Hawk Cocktail (score: 0.551)
   Type: cocktail | Base: whiskey | Strength: high
   Method: stirred 

In [23]:
# TRUE RAG - Retrieval + Generation με Claude
def ask_cocktail_expert(question: str, top_k: int = 5):
    """
    Ρωτάει τον cocktail expert (Claude) με context από το RAG.
    """
    print(f"❓ Question: {question}")
    print("=" * 70)
    
    # 1. RETRIEVAL: Βρες σχετικές συνταγές
    query_embedding = openai_client.embeddings.create(
        model="text-embedding-3-small",
        input=question
    ).data[0].embedding
    
    results = index.query(
        vector=query_embedding,
        top_k=top_k,
        include_metadata=True
    )
    
    # 2. Format context για τον Claude
    context_parts = []
    for i, match in enumerate(results['matches'], 1):
        meta = match['metadata']
        context_parts.append(
            f"[Recipe {i}: {meta['name']}]\n"
            f"Type: {meta['type']}, Base: {meta['base_spirit']}, "
            f"Strength: {meta['strength']}, Sweetness: {meta['sweetness']}\n"
            f"Text: {meta['text']}\n"
        )
    
    context = "\n".join(context_parts)
    
    # 3. GENERATION: Ρώτα τον Claude
    prompt = f"""You are a knowledgeable cocktail expert with access to a 1914 vintage cocktail book.

A user asks: "{question}"

I've retrieved the {top_k} most relevant recipes from the book:

{context}

Based on these recipes, answer the user's question in a friendly, helpful way. 
Recommend specific recipe(s) and explain why they fit. If none of the retrieved 
recipes truly match, say so honestly. Be concise but insightful.
Respond in the same language as the question."""

    response = anthropic_client.messages.create(
        model="claude-haiku-4-5",
        max_tokens=800,
        messages=[{"role": "user", "content": prompt}]
    )
    
    print("\n🤖 Claude's Answer:")
    print("-" * 70)
    print(response.content[0].text)
    print(f"\n💰 Cost: ~${(response.usage.input_tokens * 0.80 + response.usage.output_tokens * 4.00) / 1_000_000:.5f}")
    print("=" * 70)


# Δοκίμασε!
ask_cocktail_expert("Θέλω κάτι με gin που δεν είναι πολύ γλυκό, τι προτείνεις;")

❓ Question: Θέλω κάτι με gin που δεν είναι πολύ γλυκό, τι προτείνεις;

🤖 Claude's Answer:
----------------------------------------------------------------------
Καλή ερώτηση! Από τις συνταγές που έχω, η καλύτερη επιλογή για σένα είναι:

**🥃 Gin Sour** - Αυτή είναι η ιδανική επιλογή!
Έχει χαμηλή γλυκύτητα (μόνο ένα κουταλάκι ζάχαρη) και αναδεικνύει τη γεύση του gin με φρέσκο χυμό λεμονιού. Είναι κλασσική, ισορροπημένη και όχι καθόλου βαριά.

Εναλλακτικά, η **Gin Sling** θα ήταν η δεύτερη επιλογή σου - έχει μόνο ένα κομμάτι ζάχαρης και είναι απλή και κομψή.

Θα αποφύγω τις Daisy και Polly, γιατί έχουν σιρόπια που τις κάνουν πολύ γλυκές.

Καλή επιλογή! 🍋

💰 Cost: ~$0.00176


In [24]:
# Δοκιμαστικές queries — από διάφορα πλαίσια
test_queries = [
    "Ένα ρομαντικό κοκτέιλ για δείπνο δύο ατόμων",
    "Κάτι εντυπωσιακό για welcome drink σε γάμο",
    "Θέλω να δοκιμάσω κάτι με absinthe που δεν έχω ξαναδοκιμάσει",
    "Ένα cocktail με ρούμι και εξωτικά fruits",
    "What would Winston Churchill drink in 1914?",
]

for q in test_queries:
    ask_cocktail_expert(q, top_k=5)
    print("\n" + "🍸" * 35 + "\n")

❓ Question: Ένα ρομαντικό κοκτέιλ για δείπνο δύο ατόμων

🤖 Claude's Answer:
----------------------------------------------------------------------
Καλησπέρα! Για ένα ρομαντικό δείπνο δύο ατόμων, έχω δύο εξαιρετικές προτάσεις:

**1. Adalor Cup** (η καλύτερη επιλογή)
Αυτό είναι το ιδανικό επιλογή για μια ρομαντική βραδιά. Είναι κομψό, ελαφρύ και κομψό - ένα φρέσκο κύπελλο με σαμπάνια και δροσερό ροδάκινο. Τέλειο για δείπνο χωρίς να είναι υπερβολικά δυνατό.

**2. Miller Cocktail (For Two Persons)**
Σημειώστε ότι η συνταγή αυτή είναι **ειδικά για δύο άτομα**! Είναι ένα κομψό κοκτέιλ με gin, χυμό grapefruit και μαραschino - ένας καλός συνδυασμός γλυκιάς και φρέσκιας γεύσης. Δεν είναι υπερβολικά ισχυρό, ιδανικό για δείπνο.

**Η σύσταση μου:** Προτείνω το **Adalor Cup** για την απόλυτη ρομαντική ατμόσφαιρα - είναι εντυπωσιακό, κομψό και έχει μια αρχαία χάρη. Εάν προτιμάτε κάτι πιο κλασικό cocktail, το Miller είναι η σωστή επιλογή και ειδικά διαμορφωμένο για δύο άτομα! 🥂

💰 Cost: ~$0.00250

🍸🍸

In [25]:
# Advanced search με metadata filtering
def search_with_filters(
    query: str, 
    top_k: int = 5,
    base_spirit: str = None,
    max_strength: str = None,
    recipe_type: str = None,
    max_sweetness: str = None,
):
    """
    Ψάχνει με semantic search + metadata filters.
    
    Filters:
    - base_spirit: "gin", "whiskey", "rum", "brandy", "wine", etc.
    - max_strength: "low", "medium", "high"  
    - recipe_type: "cocktail", "punch", "cooler", "toddy", etc.
    - max_sweetness: "none", "low", "medium", "high"
    """
    # Build Pinecone filter (αν έχουμε criteria)
    filter_dict = {}
    if base_spirit:
        filter_dict["base_spirit"] = {"$eq": base_spirit}
    if recipe_type:
        filter_dict["type"] = {"$eq": recipe_type}
    if max_strength:
        strength_order = ["low", "medium", "high"]
        allowed = strength_order[:strength_order.index(max_strength) + 1]
        filter_dict["strength"] = {"$in": allowed}
    if max_sweetness:
        sweetness_order = ["none", "low", "medium", "high"]
        allowed = sweetness_order[:sweetness_order.index(max_sweetness) + 1]
        filter_dict["sweetness"] = {"$in": allowed}
    
    # Embed query
    query_embedding = openai_client.embeddings.create(
        model="text-embedding-3-small",
        input=query
    ).data[0].embedding
    
    # Search με filter
    results = index.query(
        vector=query_embedding,
        top_k=top_k,
        include_metadata=True,
        filter=filter_dict if filter_dict else None
    )
    
    # Print
    filter_summary = ", ".join(f"{k}={v}" for k, v in filter_dict.items()) if filter_dict else "no filters"
    print(f"\n🔍 Query: '{query}'")
    print(f"🎯 Filters: {filter_summary}")
    print("=" * 70)
    
    if not results['matches']:
        print("❌ Δεν βρέθηκαν αποτελέσματα με αυτά τα filters!")
        return results
    
    for i, match in enumerate(results['matches'], 1):
        meta = match['metadata']
        print(f"\n{i}. {meta['name']} (score: {match['score']:.3f})")
        print(f"   Type: {meta['type']} | Base: {meta['base_spirit']} | "
              f"Strength: {meta['strength']} | Sweet: {meta['sweetness']}")
    
    return results


# TEST 1: Χωρίς filters (baseline)
print("\n" + "🔥" * 35)
print("TEST 1: Χωρίς filters")
print("🔥" * 35)
search_with_filters("sweet fruity drink")

# TEST 2: Μόνο rum drinks
print("\n" + "🔥" * 35)
print("TEST 2: ΜΟΝΟ rum-based")
print("🔥" * 35)
search_with_filters("sweet fruity drink", base_spirit="rum")

# TEST 3: Low strength, καθόλου δυνατά
print("\n" + "🔥" * 35)
print("TEST 3: Light gin drinks (low/medium strength)")
print("🔥" * 35)
search_with_filters("refreshing gin drink", base_spirit="gin", max_strength="medium")


🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥
TEST 1: Χωρίς filters
🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥

🔍 Query: 'sweet fruity drink'
🎯 Filters: no filters

1. Fruit Lemonade (score: 0.605)
   Type: punch | Base: none | Strength: low | Sweet: high

2. Orangeade (score: 0.577)
   Type: cooler | Base: none | Strength: low | Sweet: medium

3. Grape Juice Cooler (score: 0.526)
   Type: cooler | Base: none | Strength: low | Sweet: medium

4. Peach Brandy Punch (score: 0.524)
   Type: punch | Base: brandy | Strength: medium | Sweet: medium

5. Brunswick Punch (score: 0.523)
   Type: other | Base: none | Strength: low | Sweet: high

🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥
TEST 2: ΜΟΝΟ rum-based
🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥

🔍 Query: 'sweet fruity drink'
🎯 Filters: base_spirit={'$eq': 'rum'}

1. Rum Daisy (score: 0.499)
   Type: cooler | Base: rum | Strength: medium | Sweet: high

2. Southern Sour (score: 0.494)
   Type: cocktail | Base: rum | Strength: medium | Sweet: medium

3. Silver Sour (score

QueryResponse(matches=[ScoredVector(id='recipe_291', score=0.628454387, values=[], metadata={'base_spirit': 'gin', 'flavor_profile': ['citrus', 'fruity', 'sweet'], 'ingredient_count': 4, 'method': 'built', 'name': 'Gin Daisy', 'strength': 'medium', 'sweetness': 'medium', 'text': '## Gin Daisy\n\nJuice 1/2 lemon.\n\n1 jigger gin.\n\n1/2 jigger raspberry syrup.\n\nIn goblet with fine ice. Fruit.', 'type': 'daisy'}), ScoredVector(id='recipe_324', score=0.59790957, values=[], metadata={'base_spirit': 'gin', 'flavor_profile': ['sweet', 'creamy', 'herbal'], 'ingredient_count': 3, 'method': 'shaken', 'name': 'Gin Flip', 'strength': 'medium', 'sweetness': 'medium', 'text': '## Gin Flip\n\n1 jigger gin.\n\n1 egg.\n\n1 barspoonful sugar. Shake well and strain.', 'type': 'cocktail'}), ScoredVector(id='recipe_403', score=0.595458269, values=[], metadata={'base_spirit': 'gin', 'flavor_profile': ['fruity', 'sweet', 'citrus', 'effervescent'], 'ingredient_count': 5, 'method': 'shaken', 'name': 'Polly'

In [27]:
import json

def intelligent_ask(question: str, top_k: int = 5):
    """
    True agentic RAG:
    1. Claude parses την ερώτηση για filters
    2. Retrieval με filters
    3. Claude generates απάντηση
    """
    print(f"❓ Question: {question}")
    print("=" * 70)
    
    # STEP 1: Claude decides filters
    filter_prompt = f"""Analyze this cocktail question and extract search filters.

Question: "{question}"

Return a JSON object with these fields (use null if not mentioned):
- base_spirit: "gin" | "whiskey" | "rum" | "brandy" | "wine" | "vermouth" | "absinthe" | null
- recipe_type: "cocktail" | "punch" | "cooler" | "fizz" | "toddy" | "cobbler" | "frappé" | null
- max_strength: "low" | "medium" | "high" | null
- max_sweetness: "none" | "low" | "medium" | "high" | null
- search_query: rewrite the question as a short semantic search query (2-6 words)

Return ONLY the JSON, no markdown, no explanation."""
    
    filter_response = anthropic_client.messages.create(
        model="claude-haiku-4-5",
        max_tokens=200,
        messages=[{"role": "user", "content": filter_prompt}]
    )
    
    try:
        filters = parse_claude_json(filter_response.content[0].text)
    except:
        filters = {"search_query": question}
    
    search_query = filters.pop("search_query", question)
    active_filters = {k: v for k, v in filters.items() if v is not None}
    
    print(f"🧠 Claude decided:")
    print(f"   Search query: '{search_query}'")
    print(f"   Filters: {active_filters if active_filters else 'none'}")
    
    # STEP 2: Build Pinecone filter
    pinecone_filter = {}
    if filters.get("base_spirit"):
        pinecone_filter["base_spirit"] = {"$eq": filters["base_spirit"]}
    if filters.get("recipe_type"):
        pinecone_filter["type"] = {"$eq": filters["recipe_type"]}
    if filters.get("max_strength"):
        strength_order = ["low", "medium", "high"]
        allowed = strength_order[:strength_order.index(filters["max_strength"]) + 1]
        pinecone_filter["strength"] = {"$in": allowed}
    if filters.get("max_sweetness"):
        sweetness_order = ["none", "low", "medium", "high"]
        allowed = sweetness_order[:sweetness_order.index(filters["max_sweetness"]) + 1]
        pinecone_filter["sweetness"] = {"$in": allowed}
    
    # STEP 3: Semantic search
    query_embedding = openai_client.embeddings.create(
        model="text-embedding-3-small",
        input=search_query
    ).data[0].embedding
    
    results = index.query(
        vector=query_embedding,
        top_k=top_k,
        include_metadata=True,
        filter=pinecone_filter if pinecone_filter else None
    )
    
    if not results['matches']:
        print("\n⚠️  Δεν βρέθηκαν αποτελέσματα με τα filters. Ξαναδοκιμάζω χωρίς filters...")
        results = index.query(
            vector=query_embedding,
            top_k=top_k,
            include_metadata=True
        )
    
    # STEP 4: Format context
    context = "\n".join([
        f"[{i}] {m['metadata']['name']} ({m['metadata']['type']}, {m['metadata']['base_spirit']}, strength: {m['metadata']['strength']}, sweet: {m['metadata']['sweetness']})\n{m['metadata']['text']}\n"
        for i, m in enumerate(results['matches'], 1)
    ])
    
    # STEP 5: Generate answer
    final_prompt = f"""You are a knowledgeable cocktail expert with access to a 1914 vintage cocktail book.

User question: "{question}"

Retrieved recipes (already filtered by your criteria):
{context}

Give a friendly, insightful recommendation. Explain WHY you're choosing specific drinks based on the recipes shown.
Respond in the same language as the question."""
    
    final_response = anthropic_client.messages.create(
        model="claude-haiku-4-5",
        max_tokens=800,
        messages=[{"role": "user", "content": final_prompt}]
    )
    
    print(f"\n🤖 Answer:")
    print("-" * 70)
    print(final_response.content[0].text)
    
    # Cost calculation - σε ξεχωριστές γραμμές
    input_tokens = filter_response.usage.input_tokens + final_response.usage.input_tokens
    output_tokens = filter_response.usage.output_tokens + final_response.usage.output_tokens
    total_cost = (input_tokens * 0.80 + output_tokens * 4.00) / 1_000_000
    
    print(f"\n💰 Total cost: ~${total_cost:.5f}")
    print("=" * 70)


# TEST!
intelligent_ask("Θέλω κάτι με gin που δεν είναι πολύ γλυκό")

❓ Question: Θέλω κάτι με gin που δεν είναι πολύ γλυκό
🧠 Claude decided:
   Search query: 'Θέλω κάτι με gin που δεν είναι πολύ γλυκό'
   Filters: none

🤖 Answer:
----------------------------------------------------------------------
# Συστάσεις για Gin χωρίς πολλή γλυκύτητα

Με βάση τις συνταγές του 1914, έχω δύο εξαιρετικές επιλογές για εσάς:

## 🥇 **Gin Sour** (Κορυφαία επιλογή)

Αυτό είναι το τέλειο ποτό για σας! Έχει **χαμηλή γλυκύτητα** και υψηλή δύναμη. Περιέχει μόνο μια μικρή ποσότητα ζάχαρης που χρησιμεύει για να ισορροπήσει τη λεμονιά, χωρίς να καταλήγει γλυκό. Είναι κλασικό, απλό και κομψό.

## 🥈 **Gin Sling** (Εναλλακτική επιλογή)

Αν θέλετε κάτι ακόμη λιγότερο γλυκό, το Gin Sling έχει **μεσαία γλυκύτητα** με ένα απλό κομμάτι ζάχαρης και πολλό πάγο. Είναι πιο δυνατό και ανακτικό.

## ❌ **Αποφύγετε:**
- Το **Polly** (πολύ γλυκό με ρόδι σιρόπι)
- Το **Gin Daisy** (μεσαία γλυκύτητα με σιρόπι βατόμουρο)

**Σύσταση:** Προτείνω το **Gin Sour** - είναι η κλασική επιλογή για όποιον θ

In [28]:
# Debug: Δες τι επιστρέφει ο Claude για το filter parsing
question = "Θέλω κάτι με gin που δεν είναι πολύ γλυκό"

filter_prompt = f"""Analyze this cocktail question and extract search filters.

Question: "{question}"

Return a JSON object with these fields (use null if not mentioned):
- base_spirit: "gin" | "whiskey" | "rum" | "brandy" | "wine" | "vermouth" | "absinthe" | null
- recipe_type: "cocktail" | "punch" | "cooler" | "fizz" | "toddy" | "cobbler" | "frappé" | null
- max_strength: "low" | "medium" | "high" | null
- max_sweetness: "none" | "low" | "medium" | "high" | null
- search_query: rewrite the question as a short semantic search query (2-6 words)

Return ONLY the JSON, no markdown, no explanation."""

response = anthropic_client.messages.create(
    model="claude-haiku-4-5",
    max_tokens=200,
    messages=[{"role": "user", "content": filter_prompt}]
)

raw = response.content[0].text
print("🔍 Raw response από Claude:")
print("=" * 60)
print(raw)
print("=" * 60)

# Try to parse
try:
    parsed = parse_claude_json(raw)
    print("\n✅ Parsed successfully:")
    for k, v in parsed.items():
        print(f"   {k}: {v}")
except Exception as e:
    print(f"\n❌ Parse error: {e}")

🔍 Raw response από Claude:
```json
{
  "base_spirit": "gin",
  "recipe_type": null,
  "max_strength": null,
  "max_sweetness": "low",
  "search_query": "gin cocktail not sweet"
}
```

❌ Parse error: name 'parse_claude_json' is not defined


In [33]:
import re
import json

def parse_claude_json(raw_text):
    """Καθαρίζει το output του Claude από markdown code blocks."""
    cleaned = raw_text.strip()
    cleaned = re.sub(r"^```(?:json)?\s*\n", "", cleaned)
    cleaned = re.sub(r"\n```\s*$", "", cleaned)
    return json.loads(cleaned)

# Test
test_response = '''```json
{
  "base_spirit": "gin",
  "max_sweetness": "low"
}
```'''

result = parse_claude_json(test_response)
print("parse_claude_json defined and working")
print(f"Test result: {result}")

parse_claude_json defined and working
Test result: {'base_spirit': 'gin', 'max_sweetness': 'low'}


In [34]:
intelligent_ask("Θέλω κάτι με gin που δεν είναι πολύ γλυκό")

❓ Question: Θέλω κάτι με gin που δεν είναι πολύ γλυκό
🧠 Claude decided:
   Search query: 'gin cocktail not sweet'
   Filters: {'base_spirit': 'gin', 'max_sweetness': 'low'}

🤖 Answer:
----------------------------------------------------------------------
# Γεια σας! 🍸

Έχω τρεις εξαιρετικές προτάσεις για σας:

## Πρώτη επιλογή: **Down Cocktail**
Αυτό είναι το ιδανικό για σας! Έχει την υψηλότερη αναλογία gin (2/3) με μόνο 1/3 Italian vermouth, οπότε είναι ξηρό και gin-forward. Το πορτοκαλόχρωμο bitters προσθέτει λεπτότητα χωρίς γλυκύτητα. Απλό, κλασικό, τέλειο.

## Δεύτερη επιλογή: **Cabinet Cocktail**
Εξίσου ξηρό (ίσες ποσότητες gin και French vermouth), με την κλασική γαλλική πολυπλοκότητα. Ελαφρώς λιγότερο gin από το Down, αλλά εξαιρετικά ισορροπημένο.

## Τρίτη επιλογή: **Silver Cocktail**
Λίγο πιο περίπλοκο με τα bitters και τη μυροδιά του maraschino, αλλά ακόμα ξηρό και εξαιρετικό αν θέλετε λίγο περισσότερη "προσωπικότητα".

**Αποφύγετε**: Το Virgin Cocktail, καθώς έχει raspberry 

In [35]:
# Ο πραγματικός Bartender Prompt
def bartender_ask(question: str, top_k: int = 5, use_filters: bool = True):
    """
    Vintage bartender expert με πλούσιες, professional απαντήσεις.
    """
    print(f"❓ Question: {question}")
    print("=" * 70)
    
    # STEP 1: Filter parsing (αν use_filters=True)
    active_filters = {}
    search_query = question
    
    if use_filters:
        filter_prompt = f"""Analyze this cocktail question and extract search filters.

Question: "{question}"

Return a JSON object with these fields (use null if not mentioned):
- base_spirit: "gin" | "whiskey" | "rum" | "brandy" | "wine" | "vermouth" | "absinthe" | null
- recipe_type: "cocktail" | "punch" | "cooler" | "fizz" | "toddy" | "cobbler" | "frappé" | null
- max_strength: "low" | "medium" | "high" | null
- max_sweetness: "none" | "low" | "medium" | "high" | null
- search_query: rewrite as short semantic search query (2-6 words)

Return ONLY the JSON, no markdown."""
        
        filter_response = anthropic_client.messages.create(
            model="claude-haiku-4-5",
            max_tokens=200,
            messages=[{"role": "user", "content": filter_prompt}]
        )
        
        try:
            filters = parse_claude_json(filter_response.content[0].text)
            search_query = filters.pop("search_query", question)
            active_filters = {k: v for k, v in filters.items() if v is not None}
        except:
            pass
    
    # STEP 2: Build Pinecone filter
    pinecone_filter = {}
    if active_filters.get("base_spirit"):
        pinecone_filter["base_spirit"] = {"$eq": active_filters["base_spirit"]}
    if active_filters.get("recipe_type"):
        pinecone_filter["type"] = {"$eq": active_filters["recipe_type"]}
    if active_filters.get("max_strength"):
        order = ["low", "medium", "high"]
        allowed = order[:order.index(active_filters["max_strength"]) + 1]
        pinecone_filter["strength"] = {"$in": allowed}
    if active_filters.get("max_sweetness"):
        order = ["none", "low", "medium", "high"]
        allowed = order[:order.index(active_filters["max_sweetness"]) + 1]
        pinecone_filter["sweetness"] = {"$in": allowed}
    
    print(f"🎯 Filters: {active_filters if active_filters else 'none'}")
    
    # STEP 3: Semantic search
    query_embedding = openai_client.embeddings.create(
        model="text-embedding-3-small",
        input=search_query
    ).data[0].embedding
    
    results = index.query(
        vector=query_embedding,
        top_k=top_k,
        include_metadata=True,
        filter=pinecone_filter if pinecone_filter else None
    )
    
    if not results['matches']:
        results = index.query(
            vector=query_embedding,
            top_k=top_k,
            include_metadata=True
        )
    
    # STEP 4: Format context
    context = "\n\n".join([
        f"[Recipe {i}] {m['metadata']['name']}\nType: {m['metadata']['type']} | Base: {m['metadata']['base_spirit']} | Method: {m['metadata']['method']} | Strength: {m['metadata']['strength']} | Sweet: {m['metadata']['sweetness']}\nFlavors: {', '.join(m['metadata'].get('flavor_profile', []))}\n{m['metadata']['text']}"
        for i, m in enumerate(results['matches'], 1)
    ])
    
    # STEP 5: Bartender-style prompt
    bartender_prompt = f"""You are Jack, a charismatic vintage bartender who has been tending bar since 1914. You have deep knowledge of classic cocktails from the golden age of American bartending. You speak with the warmth and wisdom of an old-school bartender, treating every patron like an honored guest.

A patron approaches your bar and asks: "{question}"

You have access to these recipes from your personal collection (Jacques Straub's "Drinks", 1914):

{context}

Craft a response that includes:

🍸 **RECOMMENDATION**: Your top pick, with brief reasoning about WHY it fits this specific patron and occasion
📖 **THE RECIPE**: Present it beautifully (ingredients + method)
🥃 **BARTENDER'S NOTES**: Add 1-2 professional tips that only an experienced bartender knows:
   - Suggested glassware (cocktail coupe, rocks, highball, etc.)
   - Ideal garnish
   - Best serving temperature
   - When to serve it (aperitif, digestif, summer afternoon, etc.)
   - Food pairing suggestion
   - Historical context if interesting
🎭 **ALTERNATIVE**: One backup option briefly, in case they want something different
   
Keep it warm and personal, like you're actually speaking to them across the bar. Use vintage/classic language sparingly for atmosphere. Respond in the same language as the question.

Keep it under 300 words - patrons don't want a lecture, they want a drink!"""

    final_response = anthropic_client.messages.create(
        model="claude-haiku-4-5",
        max_tokens=1000,
        messages=[{"role": "user", "content": bartender_prompt}]
    )
    
    print(f"\n🎩 Jack, the Bartender:")
    print("-" * 70)
    print(final_response.content[0].text)
    
    # Cost
    input_tokens = final_response.usage.input_tokens
    output_tokens = final_response.usage.output_tokens
    if use_filters:
        input_tokens += filter_response.usage.input_tokens
        output_tokens += filter_response.usage.output_tokens
    total_cost = (input_tokens * 0.80 + output_tokens * 4.00) / 1_000_000
    print(f"\n💰 Cost: ~${total_cost:.5f}")
    print("=" * 70)


# TEST!
bartender_ask("Θέλω κάτι με gin που δεν είναι πολύ γλυκό")

❓ Question: Θέλω κάτι με gin που δεν είναι πολύ γλυκό
🎯 Filters: {'base_spirit': 'gin', 'max_sweetness': 'low'}

🎩 Jack, the Bartender:
----------------------------------------------------------------------
*leans forward with a knowing smile*

Καλώς ήρθατε, φίλε! Gin χωρίς γλύκα - σωστή επιλογή. Ξέρω ακριβώς τι χρειάζεστε.

🍸 **ΠΡΟΣΦΟΡΑ**: **Cabinet Cocktail**

Αυτό είναι το αγαπημένό μας για τους που ξέρουν. Γαλλικό vermuth και ξηρό gin - καθαρό, ανδρικό, χωρίς περιττές γλυκάνσεις. Το πορτοκάλι σας δίνει ένα λεπτό φρέσκο άρωμα, αλλά τα bitters κρατούν τα ηνία. Τέλειο.

📖 **ΤΟ ΚΟΚΤΈΙΛ**

- 1/2 jigger γαλλικό vermuth
- 1/2 jigger ξηρό gin
- Πορτοκάλι (για το twist)

*Ταρακουχήστε με πάγο και σερβίρετε αμέσως*

🥃 **ΣΥΜΒΟΥΛΕΣ ΑΠΟ ΤΗΝ ΕΜΠΕΙΡΙΑ**

Το κλειδί εδώ είναι η **θερμοκρασία** - ψύχετε το ποτήρι πρώτα. Το cocktail coupe είναι το σωστό σκεύος, όχι τα rocks. Και όταν κάνετε το twist με το πορτοκάλι, στύψτε το σταθμό πάνω από το ποτό - το έλαιό του ανοίγει τα αρώματα. Σερβίρεται καλύτ

In [36]:
# Ας δούμε τη Jack σε πραγματικά challenging scenarios
challenging_queries = [
    "Πρώτη φορά πίνω κοκτέιλ, τι θα μου πρότεινες κάτι μαλακό;",
    "Σχεδιάζω dinner party για 8 άτομα, θέλω ένα punch που να εντυπωσιάσει",
    "Έχω φάει heavy meat, θέλω κάτι digestif",
    "Θέλω κάτι με absinthe αλλά όχι κλασικό",
]

for q in challenging_queries:
    bartender_ask(q, top_k=5)
    print("\n" + "🎩" * 35 + "\n")

❓ Question: Πρώτη φορά πίνω κοκτέιλ, τι θα μου πρότεινες κάτι μαλακό;
🎯 Filters: {'recipe_type': 'cocktail', 'max_strength': 'low', 'max_sweetness': 'high'}

🎩 Jack, the Bartender:
----------------------------------------------------------------------
*leans against the bar with a warm smile*

Καλώς ήρθες, φίλε μου! Πρώτη φορά με κοκτέιλ; Τότε έχω το τέλειο ποτό για σένα.

🍸 **ΠΡΟΤΑΣΗ**: **Isabelle Cocktail**

Ακούστε με - αυτό είναι το ιδανικό εισαγωγικό ποτό. Είναι γλυκό, εύκολο, και η κόκκινη απόχρωσή του είναι σαν μια αγκαλιά σε ένα ποτήρι. Δεν υπάρχει κανένα δυνατό αλκοόλ που θα σας τρομάξει - απλώς αρμονία και ευχαρίστηση.

📖 **ΤΟ ΣΥΝΤΑΓΜΑ**:

- 1 μικρό κομμάτι πάγου σε κοκτέιλ ποτήρι
- ½ jigger σιρόπι γρεναδίνης
- ½ jigger crème de cassis

*Χτυπήστε ελαφρά και σερβίρετε*.

🥃 **ΣΗΜΕΙΩΣΕΙΣ BARTENDER**:

Χρησιμοποιήστε ένα κλασικό **coupe glass** - το κρύο ποτήρι θα σας κάνει το ποτό να διαρκέσει. Το σημαντικό: μην το ανακατεύετε υπερβολικά. Θέλετε αυτές τις όμορφες στρώσεις χρωμάτ

In [38]:
# Correct verification για Cohere
documents = ["Hello world!", "Goodbye"]

response = co.rerank(
    model="rerank-v3.5",
    query="Say hello",
    documents=documents,
    top_n=1
)

print(f"✅ Cohere reranker works!")
print(f"   Top result index: {response.results[0].index}")
print(f"   Top result text: {documents[response.results[0].index]}")
print(f"   Relevance score: {response.results[0].relevance_score:.3f}")

✅ Cohere reranker works!
   Top result index: 0
   Top result text: Hello world!
   Relevance score: 0.328


In [39]:
# HYBRID SEARCH + RERANKING pipeline
def hybrid_search_with_rerank(
    query: str, 
    initial_top_k: int = 30,  # πάρε πολλά υποψήφια
    final_top_k: int = 5,      # rerank κρατάει τα καλύτερα
    filter_dict: dict = None,
    verbose: bool = True
):
    """
    Advanced retrieval:
    1. Wide semantic search (30 candidates)
    2. Cohere rerank → top 5
    """
    # STEP 1: Get many candidates from Pinecone
    query_embedding = openai_client.embeddings.create(
        model="text-embedding-3-small",
        input=query
    ).data[0].embedding
    
    initial_results = index.query(
        vector=query_embedding,
        top_k=initial_top_k,
        include_metadata=True,
        filter=filter_dict if filter_dict else None
    )
    
    if not initial_results['matches']:
        return []
    
    # STEP 2: Prepare documents για reranking
    # Χρησιμοποιούμε το full text της συνταγής + metadata για rich context
    documents_for_rerank = []
    for match in initial_results['matches']:
        meta = match['metadata']
        # Format: name + text + flavor context
        doc_text = f"{meta['name']}\n{meta['text']}\nType: {meta['type']}, Base: {meta['base_spirit']}, Flavors: {', '.join(meta.get('flavor_profile', []))}"
        documents_for_rerank.append(doc_text)
    
    # STEP 3: Cohere rerank
    rerank_response = co.rerank(
        model="rerank-v3.5",
        query=query,
        documents=documents_for_rerank,
        top_n=final_top_k
    )
    
    # STEP 4: Compose final results
    reranked_matches = []
    for r in rerank_response.results:
        original_match = initial_results['matches'][r.index]
        reranked_matches.append({
            'match': original_match,
            'semantic_score': original_match['score'],
            'rerank_score': r.relevance_score,
        })
    
    if verbose:
        print(f"🔍 Query: '{query}'")
        print(f"📊 Initial semantic: {len(initial_results['matches'])} candidates")
        print(f"🎯 After reranking: top {final_top_k}")
        print("=" * 70)
        
        for i, item in enumerate(reranked_matches, 1):
            m = item['match']
            meta = m['metadata']
            print(f"\n{i}. {meta['name']}")
            print(f"   Semantic: {item['semantic_score']:.3f} | Rerank: {item['rerank_score']:.3f}")
            print(f"   Type: {meta['type']} | Base: {meta['base_spirit']}")
    
    return reranked_matches


# COMPARISON: Χωρίς vs Με reranking
test_query = "sophisticated whiskey cocktail for connoisseur"

print("=" * 70)
print("🔴 ΧΩΡΙΣ RERANKING (μόνο semantic):")
print("=" * 70)

query_emb = openai_client.embeddings.create(
    model="text-embedding-3-small",
    input=test_query
).data[0].embedding

semantic_only = index.query(
    vector=query_emb,
    top_k=5,
    include_metadata=True
)

for i, match in enumerate(semantic_only['matches'], 1):
    meta = match['metadata']
    print(f"{i}. {meta['name']} (score: {match['score']:.3f}) - Base: {meta['base_spirit']}")

print("\n")
print("=" * 70)
print("🟢 ΜΕ HYBRID SEARCH + RERANKING:")
print("=" * 70)

reranked = hybrid_search_with_rerank(test_query, initial_top_k=30, final_top_k=5)

🔴 ΧΩΡΙΣ RERANKING (μόνο semantic):
1. Whiskey Cobbler (score: 0.572) - Base: whiskey
2. Holstein Cocktail (score: 0.536) - Base: brandy
3. Whiskey Float (score: 0.519) - Base: whiskey
4. Fairbank's Cocktail (score: 0.517) - Base: whiskey
5. Whiskey Cocktail (score: 0.516) - Base: whiskey


🟢 ΜΕ HYBRID SEARCH + RERANKING:
🔍 Query: 'sophisticated whiskey cocktail for connoisseur'
📊 Initial semantic: 30 candidates
🎯 After reranking: top 5

1. Red Swizzle
   Semantic: 0.507 | Rerank: 0.419
   Type: cocktail | Base: whiskey

2. Whiskey Cocktail
   Semantic: 0.516 | Rerank: 0.410
   Type: cocktail | Base: whiskey

3. Millionaire Cocktail
   Semantic: 0.497 | Rerank: 0.334
   Type: cocktail | Base: whiskey

4. Fancy Brandy Cocktail, Fancy Gin Cocktail, and Fancy Whiskey Cocktail
   Semantic: 0.497 | Rerank: 0.314
   Type: cocktail | Base: brandy, gin, or whiskey

5. Waldorf Cocktail
   Semantic: 0.500 | Rerank: 0.302
   Type: cocktail | Base: whiskey


In [40]:
# THE ULTIMATE COCKTAIL RAG - όλα τα refinements μαζί!
def ultimate_bartender(question: str, initial_top_k: int = 30, final_top_k: int = 5):
    """
    Το πληρέστερο RAG pipeline:
    1. Claude parses filters από την ερώτηση
    2. Wide semantic search (30 candidates)
    3. Cohere rerank → top 5
    4. Jack the Bartender δίνει την τελική απάντηση
    """
    print(f"❓ Question: {question}")
    print("=" * 70)
    
    # STEP 1: Parse filters
    filter_prompt = f"""Analyze this cocktail question and extract search filters.

Question: "{question}"

Return a JSON object with these fields (use null if not mentioned):
- base_spirit: "gin" | "whiskey" | "rum" | "brandy" | "wine" | "vermouth" | "absinthe" | null
- recipe_type: "cocktail" | "punch" | "cooler" | "fizz" | "toddy" | "cobbler" | "frappé" | null
- max_strength: "low" | "medium" | "high" | null
- max_sweetness: "none" | "low" | "medium" | "high" | null
- search_query: rewrite as short semantic search query (2-6 words)

Return ONLY the JSON."""
    
    filter_response = anthropic_client.messages.create(
        model="claude-haiku-4-5",
        max_tokens=200,
        messages=[{"role": "user", "content": filter_prompt}]
    )
    
    try:
        filters = parse_claude_json(filter_response.content[0].text)
        search_query = filters.pop("search_query", question)
        active_filters = {k: v for k, v in filters.items() if v is not None}
    except:
        search_query = question
        active_filters = {}
    
    # Build Pinecone filter
    pinecone_filter = {}
    if active_filters.get("base_spirit"):
        pinecone_filter["base_spirit"] = {"$eq": active_filters["base_spirit"]}
    if active_filters.get("recipe_type"):
        pinecone_filter["type"] = {"$eq": active_filters["recipe_type"]}
    if active_filters.get("max_strength"):
        order = ["low", "medium", "high"]
        allowed = order[:order.index(active_filters["max_strength"]) + 1]
        pinecone_filter["strength"] = {"$in": allowed}
    if active_filters.get("max_sweetness"):
        order = ["none", "low", "medium", "high"]
        allowed = order[:order.index(active_filters["max_sweetness"]) + 1]
        pinecone_filter["sweetness"] = {"$in": allowed}
    
    print(f"🎯 Filters: {active_filters if active_filters else 'none'}")
    print(f"🔍 Search: '{search_query}'")
    
    # STEP 2: Hybrid search + rerank
    reranked = hybrid_search_with_rerank(
        query=search_query,
        initial_top_k=initial_top_k,
        final_top_k=final_top_k,
        filter_dict=pinecone_filter if pinecone_filter else None,
        verbose=False
    )
    
    if not reranked:
        print("⚠️  Δεν βρέθηκαν αποτελέσματα με τα filters, δοκιμάζω χωρίς...")
        reranked = hybrid_search_with_rerank(
            query=search_query,
            initial_top_k=initial_top_k,
            final_top_k=final_top_k,
            verbose=False
        )
    
    # Print retrieved recipes
    print(f"\n📚 Top {len(reranked)} recipes (after reranking):")
    for i, item in enumerate(reranked, 1):
        meta = item['match']['metadata']
        print(f"   {i}. {meta['name']} (rerank: {item['rerank_score']:.3f})")
    
    # STEP 3: Format context για Jack
    context = "\n\n".join([
        f"[Recipe {i}] {item['match']['metadata']['name']}\nType: {item['match']['metadata']['type']} | Base: {item['match']['metadata']['base_spirit']} | Method: {item['match']['metadata']['method']} | Strength: {item['match']['metadata']['strength']} | Sweet: {item['match']['metadata']['sweetness']}\nFlavors: {', '.join(item['match']['metadata'].get('flavor_profile', []))}\n{item['match']['metadata']['text']}"
        for i, item in enumerate(reranked, 1)
    ])
    
    # STEP 4: Jack the Bartender
    bartender_prompt = f"""You are Jack, a charismatic vintage bartender who has been tending bar since 1914. You have deep knowledge of classic cocktails from the golden age of American bartending. You speak with the warmth and wisdom of an old-school bartender.

A patron approaches your bar and asks: "{question}"

You have access to these highly relevant recipes (already filtered and ranked by relevance):

{context}

Craft a response that includes:

🍸 **RECOMMENDATION**: Your top pick, with brief reasoning about WHY it fits
📖 **THE RECIPE**: Present it beautifully (ingredients + method)
🥃 **BARTENDER'S NOTES**: 1-2 professional tips (glassware, garnish, temperature, timing, food pairing)
🎭 **ALTERNATIVE**: One backup option briefly

Keep it warm and personal. Respond in the same language as the question. Under 300 words."""

    final_response = anthropic_client.messages.create(
        model="claude-haiku-4-5",
        max_tokens=1000,
        messages=[{"role": "user", "content": bartender_prompt}]
    )
    
    print(f"\n🎩 Jack:")
    print("-" * 70)
    print(final_response.content[0].text)
    
    # Cost
    input_tokens = filter_response.usage.input_tokens + final_response.usage.input_tokens
    output_tokens = filter_response.usage.output_tokens + final_response.usage.output_tokens
    total_cost = (input_tokens * 0.80 + output_tokens * 4.00) / 1_000_000
    print(f"\n💰 Cost: ~${total_cost:.5f}")
    print("=" * 70)


# THE ULTIMATE TEST!
ultimate_bartender("Θέλω κάτι sophisticated με whiskey για vintage lounge atmosphere")

❓ Question: Θέλω κάτι sophisticated με whiskey για vintage lounge atmosphere
🎯 Filters: {'base_spirit': 'whiskey', 'max_strength': 'high', 'max_sweetness': 'low'}
🔍 Search: 'sophisticated whiskey vintage lounge'

📚 Top 5 recipes (after reranking):
   1. Narragansett Cocktail (rerank: 0.059)
   2. Waldorf Cocktail (rerank: 0.054)
   3. Whiskey Cocktail (rerank: 0.054)
   4. Alexander Cocktail (rerank: 0.051)
   5. Black Hawk Cocktail (rerank: 0.050)

🎩 Jack:
----------------------------------------------------------------------
*leans against the bar with a knowing smile*

Καλησπέρα, φίλε. Ακούω τι ζητάς, και έχω ακριβώς αυτό που χρειάζεσαι.

🍸 **RECOMMENDATION: Narragansett Cocktail**

Αυτό είναι το ποτό της κλάσης. Όταν λες "sophisticated" με vintage lounge atmosphere, αυτό το κοκτέιλ είναι σαν μια συμφωνία στο ποτήρι σου. Ξηρό, κομψό, με εκείνη την αίσθηση της παλιάς Αμερικής που ζητάς.

📖 **THE RECIPE**

- 2/3 jigger rye whiskey
- 1/3 jigger Italian vermouth
- 1 dash absinthe
- 1 gr